# Phase 1 finalization: benchmark, error analysis, selection and inference
Run this notebook after the three sentiment notebooks have saved their models and validation prediction CSV files.

In [ ]:
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = Path("/content/voxforge-ai-review-analytics")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
%cd $PROJECT_ROOT

In [ ]:
import json
import pandas as pd
from src.config import *
from src.data.load import load_csv
from src.sentiment.comparison import benchmark_from_predictions, build_benchmark_table, generate_comparison_report, path_size_mb
from src.sentiment.error_analysis import build_error_analysis, save_error_subsets, compare_two_models
from src.sentiment.model_selection import select_production_model, save_selection
from src.sentiment.inference import SentimentPredictor, enrich_reviews
from src.sentiment.benchmarking import benchmark_predictor, save_speed_benchmark
create_project_directories()

## 1. Register completed model artifacts and prediction files
Adjust only a path if your saved filename is different.

In [ ]:
MODEL_RUNS = [
    {"model_id": "sentiment_01", "model_name": "TF-IDF Logistic Regression", "artifact": SENTIMENT_MODELS_DIR / "tfidf_logistic_regression.joblib", "predictions": PREDICTIONS_DIR / "tfidf_logistic_validation.csv"},
    {"model_id": "sentiment_02", "model_name": "DistilBERT", "artifact": SENTIMENT_MODELS_DIR / "distilbert_sentiment", "predictions": PREDICTIONS_DIR / "distilbert_validation_predictions.csv"},
    {"model_id": "sentiment_03", "model_name": "DeBERTa v3", "artifact": SENTIMENT_MODELS_DIR / "deberta_v3_sentiment", "predictions": PREDICTIONS_DIR / "deberta_v3_sentiment_validation.csv"},
]
tracker = pd.read_csv(MODEL_TRACKING_PATH) if MODEL_TRACKING_PATH.exists() else pd.DataFrame()
MODEL_RUNS

In [ ]:
rows = []
loaded_predictions = {}
for item in MODEL_RUNS:
    if not item["predictions"].exists():
        print("Skipping missing predictions:", item["predictions"])
        continue
    predictions = pd.read_csv(item["predictions"])
    loaded_predictions[item["model_id"]] = predictions
    tracked = tracker.loc[tracker["model_id"] == item["model_id"]].tail(1) if not tracker.empty else pd.DataFrame()
    rows.append(benchmark_from_predictions(
        predictions, model_id=item["model_id"], model_name=item["model_name"],
        training_time_seconds=None if tracked.empty else tracked.iloc[0].get("training_time_seconds"),
        inference_time_ms=None if tracked.empty else tracked.iloc[0].get("inference_time_ms"),
        model_size_mb=path_size_mb(item["artifact"]),
    ))
benchmark = build_benchmark_table(rows, MODEL_BENCHMARK_PATH)
display(benchmark)

## 2. Error analysis

In [ ]:
for item in MODEL_RUNS:
    if item["model_id"] not in loaded_predictions:
        continue
    analysis = build_error_analysis(loaded_predictions[item["model_id"]])
    save_error_subsets(analysis, ERROR_ANALYSIS_DIR, item["model_id"])
    display(analysis.loc[~analysis["is_correct"]].head(10))

In [ ]:
if "sentiment_02" in loaded_predictions and "sentiment_03" in loaded_predictions:
    comparisons = compare_two_models(loaded_predictions["sentiment_02"], loaded_predictions["sentiment_03"], first_name="distilbert", second_name="deberta")
    for name, frame in comparisons.items():
        frame.to_csv(ERROR_ANALYSIS_DIR / f"{name}.csv", index=False)
        print(name, len(frame))

## 3. Select the production model and write the report

In [ ]:
winner, reason = select_production_model(benchmark)
selected_run = next(item for item in MODEL_RUNS if item["model_id"] == winner["model_id"])
save_selection(winner, reason, artifact_path=selected_run["artifact"], output_path=MODEL_SELECTION_PATH)
generate_comparison_report(benchmark, selected_model=winner["model_name"], reason=reason, output_path=MODEL_COMPARISON_REPORT_PATH)
print("Winner:", winner["model_name"])
print(reason)

## 4. Speed benchmark for the selected model

In [ ]:
reviews = load_csv(CLEANED_REVIEWS_PATH)
text_column = "transformer_text" if selected_run["artifact"].is_dir() else "classical_text"
predictor = SentimentPredictor(selected_run["artifact"])
speed = benchmark_predictor(predictor, reviews[text_column].dropna().head(1000), model_name=winner["model_name"], batch_sizes=(1, 8, 32))
save_speed_benchmark([speed], INFERENCE_SPEED_PATH)
display(speed)

## 5. Run production inference on the complete cleaned dataset

In [ ]:
enriched = enrich_reviews(reviews, predictor, text_column=text_column, output_path=ENRICHED_REVIEWS_PATH)
print(f"Saved {len(enriched):,} enriched reviews to {ENRICHED_REVIEWS_PATH}")
display(enriched.head())

## Ablation study note
The reusable tracker is in `src/sentiment/ablation.py`. Run only controlled retraining experiments that are useful: class weights on/off and max length 128/256. Do not block Phase 2 on ablations.